In [1]:
import os
import json
import glob
import numpy as np
import chipwhisperer as cw

from typing import List
from chipwhisperer.common.traces import Trace
from chipwhisperer.common.utils.util import CWByteArray
from chipwhisperer.common.api.ProjectFormat import Project

/Users/xcrbox/Desktop/2026-eCTF/chipwhisperer/jupyter/.venv/lib/python3.10/site-packages/chipwhisperer/capture/trace/TraceWhisperer.py:31: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources # type: ignore


In [35]:
class DataObject:
    """Basically a mirror of the CW `Trace` object with an additional settings added in.
    """
    def __init__(self, wave, textin, textout, settings={}, key=""):
        self.wave = wave
        self.textin = textin
        self.textout = textout
        self.key = key
        self.settings = settings # Considering removing this field because it leads to redundant python storage

    @classmethod
    def empty(cls):
        return cls([], b"", b"", b"", {}, b"")

    @classmethod
    def from_trace(cls, trace_object: Trace, settings={}):
        return cls(trace_object.wave,
                   trace_object.textin,
                   trace_object.textout,
                   settings,
                   trace_object.key)

    def __str__(self):
        return f'Wave len: {len(self.wave)}; '\
            f'Textin (hex): [{" ".join([f"{textin:02x}".upper() for textin in self.textin])}]; '\
            f'Textout (hex): [{" ".join([f"{textout:02x}".upper() for textout in self.textout])}]; '\
            f'Settings: {self.settings}'

    def __repr__(self):
        return str(self)

    def __sub__(self, other):
        self.wave = self.wave - other.wave
        return self

    def constrain_wave(self, start, stop):
        self.wave = self.wave[start:stop]
        self.settings['num_samples'] = stop - start
        return self

    def cwTrace(self) -> Trace:
        return Trace(self.wave, CWByteArray(self.textin), CWByteArray(self.textout), CWByteArray(self.key))

In [27]:
def np_lists(trace_object_list: List[DataObject]) -> np.ndarray:
    """Converts list of `DataObject` to a numpy array that can be used for quick numpy computations.
    """
    return np.array([trace_item.wave for trace_item in trace_object_list])

In [32]:
def data_to_proj(proj: Project, trace_object_list: List[DataObject]):
    """Converts a list of `DataObject` into a chipwhisperer project.
    """
    for trace_object in trace_object_list:
        proj.traces.append(trace_object.cwTrace())
    return

In [29]:
def print_data_array(trace_object_list: List[DataObject]) -> None:
    print('\n'.join([str(trace_item) for trace_item in trace_object_list]))
    return

In [72]:
def save_data(lab_dirname: str, trace_object_list: List[Trace], settings={}, overwrite=False) -> None:
    """Saves a list of traces to a directory within the `./data/` directory.
    """
    cwd = os.getcwd()
    new_directory = f'{cwd}/data/{lab_dirname}'
    settings['num_entries'] = len(trace_object_list)
    text_list = []

    
    if not os.path.isdir(new_directory):
        print(f'Creating new data directory at "{new_directory}"')
        os.makedirs(new_directory, exist_ok=True)
    elif not overwrite:
        print(f'Directory already exists at "{new_directory}"')
        return

    text_list.append('index,key,textin,textout\n')
    for trace_index in range(len(trace_object_list)):
        text_list.append(f'{trace_index},'\
                         f'{(trace_object_list[trace_index].key or b"").hex()},'\
                         f'{(trace_object_list[trace_index].textin or b"").hex()},'\
                         f'{(trace_object_list[trace_index].textout or b"").hex()}\n')
        np.save(f'{new_directory}/trace_{trace_index}.npy', np.array(trace_object_list[trace_index].wave))

    with open(f'{new_directory}/text.csv', 'w', encoding='ascii') as file:
        file.writelines(text_list)

    with open(f'{new_directory}/settings.json', 'w', encoding='ascii') as file:
        json.dump(settings, file)

In [74]:
def import_data(lab_dirname: str) -> (List[Trace], dict):
    """Imports list of trace objects from the `./data/` directory.
    """
    cwd = os.getcwd()
    new_directory = f'{cwd}/data/{lab_dirname}'
    settings_json_path = f'{new_directory}/settings.json'
    text_csv_path = f'{new_directory}/text.csv'
    settings = {}
    trace_object_list = []

    assert os.path.isdir(new_directory)
    assert os.path.isfile(settings_json_path)
    assert os.path.isfile(text_csv_path)

    with open(settings_json_path) as file:
        settings = json.load(file)
        
    with open(text_csv_path) as file:
        text_list = file.readlines()
    
    for trace_index in range(settings['num_entries']):
        np_wave = np.load(f'{new_directory}/trace_{trace_index}.npy')
        text_split_list = text_list[trace_index+1].split(',')
        key = bytes.fromhex(text_split_list[1])
        textin = bytes.fromhex(text_split_list[2])
        textout = bytes.fromhex(text_split_list[3])
        trace_object_list.append(Trace(np_wave, CWByteArray(textin), CWByteArray(textout), CWByteArray(key)))

    print(f'Number of Traces: {settings["num_entries"]}')
    return trace_object_list, settings

In [10]:
def clear_data(lab_name_pattern: str) -> None:
    for file in glob.glob(lab_name_pattern):
        os.remove(file)

In [11]:
def save_record(lab_filename: str, record: List[str]) -> None:
    cwd = os.getcwd()
    file = open(f'{cwd}/data/{lab_filename}', 'w')
    file.writelines([record_item + '\n' for record_item in record])
    file.close()

In [12]:
def import_record(lab_filename: str) -> List[str]:
    cwd = os.getcwd()
    file = open(f'{cwd}/data/{lab_filename}', 'r')
    file_lines = file.readlines()
    file.close()
    return file_lines